<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales — Modelo 5: Matriz Cruzada ABC / XYZ Segmentation

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Responsable: **Jack (Lead / Arquitecto Big Data)**.

Nota crítica de esquema: `../silver/merge_pyspark` **no tiene columna `quantity` ni stock**; cada fila representa **1 unidad vendida**. Por ello:

- **ABC (Pareto de Ingresos):** participación acumulada de `SUM(price)` (`A <= 80%`, `B <= 95%`, `C > 95%`).
- **XYZ (Estabilidad de Demanda):** Coeficiente de Variación (`CV = stddev / mean`) de unidades mensuales **sobre la grilla completa `article_id x mes`** (imputando `0` en meses sin venta para evitar sesgo): `X <= 0.50`, `Y <= 1.00`, `Z > 1.00`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder
    .appName("HYM_Lab13_05_ABC_XYZ")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")
spark

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Rutas de capas

In [ ]:
SILVER = "../silver"
GOLD   = "../gold"

import os
os.makedirs(GOLD, exist_ok=True)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Lectura desde Silver y grilla completa artículo × mes

In [ ]:
%%time
df_silver = (
    spark.read.parquet(f"{SILVER}/merge_pyspark")
    .select("article_id", "prod_name", "product_group_name", "department_name", "Fecha", "price")
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    .withColumn("mes_venta", F.date_trunc("MONTH", F.col("Fecha")))
)

df_meses = df_silver.select("mes_venta").distinct()
n_meses = df_meses.count()

df_cat = df_silver.groupBy("article_id").agg(
    F.first("prod_name", ignorenulls=True).alias("prod_name"),
    F.first("product_group_name", ignorenulls=True).alias("product_group_name"),
    F.first("department_name", ignorenulls=True).alias("department_name"),
    F.sum("price").alias("ingreso_total"),
    F.count("*").alias("unidades_totales"),
)

df_ventas_mes = df_silver.groupBy("article_id", "mes_venta").agg(F.count("*").cast("double").alias("unidades_mes"))

# CrossJoin ligero de catálogo x meses para completar ceros sin inflar con toPandas
df_grilla = (
    df_cat.select("article_id")
    .crossJoin(F.broadcast(df_meses))
    .join(df_ventas_mes, on=["article_id", "mes_venta"], how="left")
    .fillna(0.0, subset=["unidades_mes"])
)
print(f"Artículos únicos: {df_cat.count():,} | Meses calendario: {n_meses}")

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Clasificación ABC (Pareto ingresos) y XYZ (CV sobre grilla completa)

In [ ]:
%%time
# XYZ por artículo
df_xyz = (
    df_grilla.groupBy("article_id")
    .agg(
        F.avg("unidades_mes").alias("media_mensual"),
        F.stddev_pop("unidades_mes").alias("std_mensual"),
    )
    .withColumn("cv_demanda", F.when(F.col("media_mensual") > 0, F.col("std_mensual") / F.col("media_mensual")).otherwise(F.lit(99.0)))
    .withColumn(
        "clase_xyz",
        F.when(F.col("cv_demanda") <= 0.50, "X")
         .when(F.col("cv_demanda") <= 1.00, "Y")
         .otherwise("Z"),
    )
)

# ABC por participación acumulada de ingreso_total
ingreso_global = df_cat.agg(F.sum("ingreso_total")).collect()[0][0]
w_abc = Window.orderBy(F.col("ingreso_total").desc()).rowsBetween(Window.unboundedPreceding, 0)

df_abc_xyz = (
    df_cat
    .withColumn("pct_ingreso_acum", F.sum("ingreso_total").over(w_abc) / F.lit(ingreso_global))
    .withColumn(
        "clase_abc",
        F.when(F.col("pct_ingreso_acum") <= 0.80, "A")
         .when(F.col("pct_ingreso_acum") <= 0.95, "B")
         .otherwise("C"),
    )
    .join(df_xyz, on="article_id", how="inner")
    .withColumn("matriz_abc_xyz", F.concat(F.col("clase_abc"), F.col("clase_xyz")))
).checkpoint()

matriz_res = (
    df_abc_xyz.groupBy("matriz_abc_xyz", "clase_abc", "clase_xyz")
    .agg(
        F.count("*").alias("skus"),
        F.sum("unidades_totales").alias("unidades"),
        F.round(F.sum("ingreso_total"), 2).alias("ingresos"),
    )
    .orderBy("clase_abc", "clase_xyz")
)
matriz_res.show(12, truncate=False)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Heatmap 3x3 ABC/XYZ y guardado en Gold

In [ ]:
pdf_m = matriz_res.toPandas()
piv_ing = pdf_m.pivot(index="clase_abc", columns="clase_xyz", values="ingresos").reindex(index=["A", "B", "C"], columns=["X", "Y", "Z"]).fillna(0)
piv_sku = pdf_m.pivot(index="clase_abc", columns="clase_xyz", values="skus").reindex(index=["A", "B", "C"], columns=["X", "Y", "Z"]).fillna(0)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
im = ax.imshow(piv_ing.values, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(["X (Estable)", "Y (Variable)", "Z (Errático)"])
ax.set_yticks(range(3)); ax.set_yticklabels(["A (80%)", "B (15%)", "C (5%)"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{piv_ing.index[i]}{piv_ing.columns[j]}\n{int(piv_sku.iloc[i, j]):,} SKUs", ha="center", va="center", color="black", fontweight="bold")
ax.set_title("Matriz Cruzada ABC / XYZ — Distribución de SKUs")
plt.tight_layout()
plt.show()

df_abc_xyz.write.mode("overwrite").parquet(f"{GOLD}/hym_abc_xyz")
print(f"Matriz ABC/XYZ guardada en {GOLD}/hym_abc_xyz | SKUs={spark.read.parquet(f'{GOLD}/hym_abc_xyz').count():,}")

In [ ]:
spark.stop()